# 09. AI Analysis: Loss Landscape Geometry, Hessian Lanczos SLQ, and Empirical NTK

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Li et al. (2018 *Visualizing the Loss Landscape of Neural Nets*); Ghorbani et al. (2019 *An Investigation into Neural Net Optimization via Hessian Energy Landscapes*); Jacot, Gabriel, & Hongler (2018 *Neural Tangent Kernel*).
- **Purpose:** Analyze curvature and training dynamics using filter-normalized loss landscape slicing, Stochastic Lanczos Quadrature (SLQ) Hessian spectral density estimation, and Empirical Neural Tangent Kernel (eNTK) Gram matrices.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [08. AI Analysis Neural Collapse Superposition and WeightWatcher RMT](08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.ipynb) $\to$ `🔗 Downstream Syntheses:` [10. AI Evaluation Science Bradley Terry IRT Dawid Skene and Min-K%](10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not visualize loss landscapes using raw unnormalized directions in networks with Batch Normalization (scale invariance creates spurious sharpness illusions); avoid running Lanczos iterations beyond $m > 20$ without reorthogonalization due to numerical rank loss.

## Mathematical Foundations & Governing Equations

1. **Filter Normalization**: $\delta_{l, j} \leftarrow \delta_{l, j} \cdot \frac{\|\theta_{l, j}\|_F}{\|\delta_{l, j}\|_F + \epsilon}$
2. **Loss Surface Slicing**: $\theta(\alpha, \beta) = \theta + \alpha \delta + \beta \eta$
3. **Lanczos Krylov Recurrence**: $w_j = H q_j - \beta_{j-1} q_{j-1}, \ \alpha_j = \langle q_j, w_j \rangle, \ \beta_j = \|w_j - \alpha_j q_j\|_2$
4. **Empirical NTK**: $\Theta = J J^T \in \mathbb{R}^{N \times N}, \quad J_i = \nabla_\theta f(x_i; \theta)$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_filter_normalize(theta, delta):
    th_norm = torch.sqrt((theta * theta).sum(dim=-1, keepdim=True) + 1e-8)
    dt_norm = torch.sqrt((delta * delta).sum(dim=-1, keepdim=True) + 1e-8)
    return delta * (th_norm / dt_norm)

def calc_loss_surface_coord(theta, d1, d2, alpha=0.5, beta=-0.25):
    return theta + float(alpha) * d1 + float(beta) * d2

def calc_lanczos(h, v0, num_steps=3):
    p = h.shape[0]
    q = [v0 / torch.sqrt((v0 * v0).sum() + 1e-8)]
    alpha_list, beta_list = [], []
    for j in range(num_steps):
        w = h @ q[j]
        if j > 0: w = w - beta_list[j - 1] * q[j - 1]
        a_j = (q[j] * w).sum()
        alpha_list.append(a_j)
        w = w - a_j * q[j]
        if j < num_steps - 1:
            b_j = torch.sqrt((w * w).sum() + 1e-8)
            beta_list.append(b_j)
            q.append(w / b_j)
    return torch.stack(alpha_list), torch.stack(beta_list)

def calc_empirical_ntk(jacobians):
    return jacobians @ jacobians.t()

In [ ]:
# Parity verification against golden reference tensors
d1_norm = calc_filter_normalize(weights["ls_theta"], weights["ls_d1_raw"])
assert torch.max(torch.abs(d1_norm - weights["ls_d1_norm"])) < 1e-5, "d1_norm mismatch"

d2_norm = calc_filter_normalize(weights["ls_theta"], weights["ls_d2_raw"])
assert torch.max(torch.abs(d2_norm - weights["ls_d2_norm"])) < 1e-5, "d2_norm mismatch"

ls_slice = calc_loss_surface_coord(weights["ls_theta"], d1_norm, d2_norm, 0.5, -0.25)
assert torch.max(torch.abs(ls_slice - weights["ls_slice_ref"])) < 1e-5, "ls_slice mismatch"

alpha_diag, beta_diag = calc_lanczos(weights["h_mat"], weights["v0"], num_steps=3)
assert torch.max(torch.abs(alpha_diag - weights["lanczos_alpha_ref"])) < 1e-5, "alpha_diag mismatch"
assert torch.max(torch.abs(beta_diag - weights["lanczos_beta_ref"])) < 1e-5, "beta_diag mismatch"

ntk = calc_empirical_ntk(weights["ntk_j"])
assert torch.max(torch.abs(ntk - weights["ntk_gram_ref"])) < 1e-5, "NTK Gram mismatch"
print("All Loss Landscape, Lanczos, and Empirical NTK assertions passed!")

In [ ]:
# Benchmarking analysis operators
t0 = time.perf_counter()
for _ in range(500):
    d1_n = calc_filter_normalize(weights["ls_theta"], weights["ls_d1_raw"])
    _ = calc_loss_surface_coord(weights["ls_theta"], d1_n, d1_n, 0.5, -0.25)
    _ = calc_lanczos(weights["h_mat"], weights["v0"], num_steps=3)
    _ = calc_empirical_ntk(weights["ntk_j"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused FilterNorm + Lanczos + NTK latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Scale-Invariance Illusion in Unnormalized vs Filter-Normalized Slicing
theta_orig = torch.tensor([[1.0, 1.0]])
theta_scaled = 10.0 * theta_orig  # Scale up weights by 10x (e.g., pre-BatchNorm scale shift)
delta_raw = torch.tensor([[0.1, 0.1]])

# Unnormalized relative step: 10x larger perturbation on theta_orig vs theta_scaled
rel_step_unnorm_orig = (torch.norm(delta_raw) / torch.norm(theta_orig)).item()
rel_step_unnorm_scaled = (torch.norm(delta_raw) / torch.norm(theta_scaled)).item()

# Filter-normalized relative step is strictly scale-invariant
delta_norm_orig = calc_filter_normalize(theta_orig, delta_raw)
delta_norm_scaled = calc_filter_normalize(theta_scaled, delta_raw)
rel_step_norm_orig = (torch.norm(delta_norm_orig) / torch.norm(theta_orig)).item()
rel_step_norm_scaled = (torch.norm(delta_norm_scaled) / torch.norm(theta_scaled)).item()
print(f"Unnormalized relative perturbation: orig={rel_step_unnorm_orig:.3f}, scaled={rel_step_unnorm_scaled:.3f}")
print(f"Filter-normalized relative perturbation: orig={rel_step_norm_orig:.3f}, scaled={rel_step_norm_scaled:.3f}")
assert abs(rel_step_norm_orig - rel_step_norm_scaled) < 1e-5, "Filter-normalization must enforce scale invariance!"
assert rel_step_unnorm_orig > 5.0 * rel_step_unnorm_scaled, "Unnormalized slicing creates false sharpness differences!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Filter-Normalized Slicing** prevents spurious curvature illusions caused by parameter rescaling under Batch Normalization.
2. **Stochastic Lanczos Quadrature** efficiently approximates the full Hessian spectral density using only matrix-vector products.
3. **Empirical NTK Gram Matrices** reveal whether model optimization operates in the linear kernel regime or the feature-learning rich regime.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.md](../../markdown/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.md)
- **PyTorch 2.x Module:** [09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.py](../../pytorch2/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.py)
- **Functional JAX Module:** [09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.py](../../jax/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.py)
- **Native Rust Candle Track:** [09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.rs](../../candle/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.rs)
- **Hardware Kernel (CPU / WGSL):** [09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.rs](../../kernels/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.rs)
- **Golden SafeTensors:** [09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.safetensors](../../weights/10_xai_and_ai_analysis_methods/09_ai_analysis_loss_landscape_hessian_lanczos_and_empirical_ntk.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 08. AI Analysis Neural Collapse Superposition and WeightWatcher RMT](08_ai_analysis_neural_collapse_superposition_and_weightwatcher_rmt.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 10. AI Evaluation Science Bradley Terry IRT Dawid Skene and Min-K%](10_ai_evaluation_science_bradley_terry_irt_dawid_skene_and_mink.ipynb) &rarr;